In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBClassifier
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
df=pd.read_csv('/content/WA_Fn-UseC_-Telco-Customer-Churn (1).csv',index_col='customerID')
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
customerID,,,,,,,,,,,,,,,,,,,,
7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7043 entries, 7590-VHVEG to 3186-AJIEK
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   object 
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   object 
 3   Dependents        7043 non-null   object 
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   object 
 6   MultipleLines     7043 non-null   object 
 7   InternetService   7043 non-null   object 
 8   OnlineSecurity    7043 non-null   object 
 9   OnlineBackup      7043 non-null   object 
 10  DeviceProtection  7043 non-null   object 
 11  TechSupport       7043 non-null   object 
 12  StreamingTV       7043 non-null   object 
 13  StreamingMovies   7043 non-null   object 
 14  Contract          7043 non-null   object 
 15  PaperlessBilling  7043 non-null   object 
 16  PaymentMethod     7043 non-null 

In [ ]:
df['Churn'].value_counts(normalize=True)*100

,proportion
Churn,
No,73.463013
Yes,26.536987


In [ ]:
df.select_dtypes('object').columns

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod', 'TotalCharges', 'Churn'],
      dtype='object')

In [ ]:
df.drop('PaymentMethod',axis=1,inplace=True)

In [ ]:
df.select_dtypes('object')

,gender,Partner,Dependents,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,TotalCharges,Churn
customerID,,,,,,,,,,,,,,,,
7590-VHVEG,Female,Yes,No,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,29.85,No
5575-GNVDE,Male,No,No,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,1889.5,No
3668-QPYBK,Male,No,No,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,108.15,Yes
7795-CFOCW,Male,No,No,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,1840.75,No
9237-HQITU,Female,No,No,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,151.65,Yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6840-RESVB,Male,Yes,Yes,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,1990.5,No
2234-XADUH,Female,Yes,Yes,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,7362.9,No
4801-JZAZL,Female,Yes,Yes,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,346.45,No


In [ ]:
df['MultipleLines'].replace({'No phone srvice':0,'No':1,'Yes':2},inplace=True)

In [ ]:
df.select_dtypes('object').columns

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'TotalCharges', 'Churn'],
      dtype='object')

In [ ]:
binary_categorical=['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
        'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies',
       'PaperlessBilling', 'TotalCharges', 'Churn']
multiclass_categorical=['InternetService','Contract']

In [ ]:
le=LabelEncoder()

In [ ]:
# Fix the 'MultipleLines' column due to mixed types (int and str) caused by a typo in previous replacement.
# Ensure 'No phone service' (correct spelling) is replaced by 0.
# This will make the 'MultipleLines' column uniformly numeric (0, 1, 2).
df['MultipleLines'].replace({'No phone service': 0}, inplace=True)

# Convert 'TotalCharges' to numeric, coercing errors to NaN.
# This handles cases where 'TotalCharges' might contain non-numeric strings (e.g., ' ').
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Impute missing values in 'TotalCharges' with the mean.
# This handles NaNs introduced by the 'errors="coerce"' step.
if df['TotalCharges'].isnull().any():
    df['TotalCharges'].fillna(df['TotalCharges'].mean(), inplace=True)

# Filter out 'TotalCharges' from binary_categorical as it's a numeric column and should not be label encoded.
binary_categorical_filtered = [col for col in binary_categorical if col != 'TotalCharges']

# Apply LabelEncoder to the remaining binary categorical columns.
for col in binary_categorical_filtered:
  # Ensure the column is of string type for consistent LabelEncoding of categorical values.
  # This helps prevent TypeErrors if columns contain mixed types (e.g., object dtype with NaNs treated as floats).
  df[col] = df[col].astype(str)
  df[col]=le.fit_transform(df[col])

In [ ]:
df=pd.get_dummies(df,columns=multiclass_categorical,drop_first=True,dtype=int)

In [ ]:
df.dtypes


,0
gender,int64
SeniorCitizen,int64
Partner,int64
Dependents,int64
tenure,int64
PhoneService,int64
MultipleLines,int64
OnlineSecurity,int64
OnlineBackup,int64
DeviceProtection,int64


In [ ]:
x=df.drop('Churn',axis=1)
y=df.Churn

In [ ]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
import xgboost as xgb
model=xgb.XGBClassifier()
model.fit(x_train,y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
y_pred_test=model.predict(x_test)
print('Classification report: ',classification_report(y_test,y_pred_test))
print('Confusion matrix: ',confusion_matrix(y_test,y_pred_test))
print('Accuracy score: ',accuracy_score(y_test,y_pred_test))

Classification report:                precision    recall  f1-score   support

           0       0.83      0.88      0.85      1035
           1       0.60      0.50      0.55       374

    accuracy                           0.78      1409
   macro avg       0.72      0.69      0.70      1409
weighted avg       0.77      0.78      0.77      1409

Confusion matrix:  [[910 125]
 [186 188]]
Accuracy score:  0.7792760823278921


In [ ]:
y_pred_train=model.predict(x_train)
print('Classification report: ',classification_report(y_train,y_pred_train))
print('Confusion matrix: ',confusion_matrix(y_train,y_pred_train))
print('Accuracy score: ',accuracy_score(y_train,y_pred_train))

Classification report:                precision    recall  f1-score   support

           0       0.95      0.97      0.96      4139
           1       0.90      0.85      0.87      1495

    accuracy                           0.94      5634
   macro avg       0.92      0.91      0.92      5634
weighted avg       0.93      0.94      0.93      5634

Confusion matrix:  [[3998  141]
 [ 224 1271]]
Accuracy score:  0.9352147674831381
